# Sta-RU — XTTS-v2 Segment Auditor

Genera un audio por línea del SRT y te los pone en fila para escucharlos uno por uno.

Útil para:
- Detectar qué líneas salen robotizadas o raras
- Comparar el mismo SRT con distintas voice refs
- Probar distintas temperaturas

## 1. GPU check

In [ ]:
!nvidia-smi | head -20

## 2. Instalar dependencias

In [ ]:
!apt-get -qq install -y ffmpeg
!pip install -q TTS srt soundfile torch ipywidgets

## 3. Clonar repo

In [ ]:
import os, sys
REPO_DIR = '/content/Sta-RU'
BRANCH = 'claude/compassionate-knuth-M2llL'
if not os.path.exists(REPO_DIR):
    !git clone --depth 1 -b $BRANCH https://github.com/lazy-money/sta-ru.git $REPO_DIR
else:
    !cd $REPO_DIR && git pull --quiet
print('Repo listo:', REPO_DIR)

## 4. Subir voice reference

WAV limpio, 6–15 segundos, mono, 44100 Hz, 16 bit PCM.

In [ ]:
from google.colab import files
from IPython.display import Audio, display
import soundfile as sf

print('Seleccioná tu archivo WAV...')
uploaded = files.upload()
VOICE_REF = '/content/' + list(uploaded.keys())[0]
with open(VOICE_REF, 'wb') as f:
    f.write(list(uploaded.values())[0])

audio_data, sr = sf.read(VOICE_REF)
dur = len(audio_data) / sr
ch = 'mono' if audio_data.ndim == 1 else f'{audio_data.shape[1]} canales'
print(f'Voice ref: {VOICE_REF}')
print(f'  Duración: {dur:.1f}s  |  Sample rate: {sr} Hz  |  {ch}')
display(Audio(VOICE_REF))

## 5. Subir SRT

In [ ]:
import srt, re

print('Seleccioná tu archivo SRT...')
uploaded_srt = files.upload()
SRT_NAME = list(uploaded_srt.keys())[0]
SRT_PATH = '/content/' + SRT_NAME
with open(SRT_PATH, 'wb') as f:
    f.write(list(uploaded_srt.values())[0])

with open(SRT_PATH, encoding='utf-8') as f:
    SUBS = list(srt.parse(f.read()))

SUBS = [s for s in SUBS if s.content.strip()]
total_dur = SUBS[-1].end.total_seconds() if SUBS else 0
print(f'SRT: {SRT_NAME}')
print(f'  {len(SUBS)} segmentos  |  duración hasta: {int(total_dur//60)}:{int(total_dur%60):02d}')
print(f'  Chars promedio por segmento: {sum(len(s.content) for s in SUBS)//max(1,len(SUBS))}')

## 6. Opciones

In [ ]:
import ipywidgets as W
from IPython.display import display

lang_w  = W.Dropdown(
    options=['en','es','fr','de','it','pt','pl','tr','ru','nl','cs','ar','zh','hu','ko','ja','hi'],
    value='en', description='Idioma:')
temp_w  = W.FloatSlider(value=0.75, min=0.1, max=1.0, step=0.05, description='Temperatura:',
                        style={'description_width': 'initial'})
range_w = W.Text(value='all', description='Rango (N°):',
                 placeholder="'all', '1-20', '5' — por N° de segmento SRT",
                 layout=W.Layout(width='500px'))

display(lang_w, temp_w, range_w)
print()
print('Temperatura: 0.75 es el default de XTTS. Subir → más variado/expresivo, bajar → más estable/robótico.')
print('Rango: filtrá qué segmentos generar. Útil para probar solo los que sabés que salen mal.')

In [ ]:
# Fijar configuración
import pandas as pd

LANG = lang_w.value
TEMPERATURE = temp_w.value
RANGE_EXPR = range_w.value.strip()

def _in_range(idx, expr):
    if expr == 'all':
        return True
    for part in expr.split(','):
        part = part.strip()
        if '-' in part:
            a, b = part.split('-', 1)
            if int(a) <= idx <= int(b):
                return True
        elif part.isdigit() and int(part) == idx:
            return True
    return False

ACTIVE_SUBS = [s for s in SUBS if _in_range(s.index, RANGE_EXPR)]

rows = []
for s in ACTIVE_SUBS[:10]:
    t = s.content.strip().replace('\n', ' ')
    rows.append({'N°': s.index,
                 'Inicio': str(s.start).split('.')[0],
                 'Fin': str(s.end).split('.')[0],
                 'Chars': len(t),
                 'Texto': t[:80]})
print(f'Configuración: lang={LANG}, temperatura={TEMPERATURE}, rango={RANGE_EXPR}')
print(f'Segmentos a generar: {len(ACTIVE_SUBS)}{" (mostrando primeros 10)" if len(ACTIVE_SUBS) > 10 else ""}')
pd.DataFrame(rows).style.hide(axis='index')

## 7. Generar segmentos

Corre XTTS-v2 sobre cada línea del SRT y guarda los WAV en `/content/xtts_out/`.

In [ ]:
import os, torch, soundfile as sf
from pathlib import Path
from tqdm.notebook import tqdm
from TTS.api import TTS

OUT_DIR = Path('/content/xtts_out')
OUT_DIR.mkdir(exist_ok=True)

os.environ['COQUI_TOS_AGREED'] = '1'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Cargando XTTS-v2 en {device}...')
tts = TTS('tts_models/multilingual/multi-dataset/xtts_v2').to(device)
print('Modelo listo.\n')

results = []   # list of (sub, path, error)
n_ok = n_fail = 0

for sub in tqdm(ACTIVE_SUBS, desc='Generando', unit='seg'):
    text = sub.content.strip().replace('\n', ' ')
    out_path = OUT_DIR / f'seg_{sub.index:04d}.wav'
    txt_path = OUT_DIR / f'seg_{sub.index:04d}.txt'
    err = None
    try:
        tts.tts_to_file(
            text=text,
            file_path=str(out_path),
            speaker_wav=VOICE_REF,
            language=LANG,
            split_sentences=False,
            temperature=TEMPERATURE,
        )
        txt_path.write_text(text, encoding='utf-8')
        n_ok += 1
    except Exception as e:
        err = str(e)
        n_fail += 1
    results.append((sub, out_path if err is None else None, err))

print(f'\nListo: {n_ok} ok, {n_fail} fallidos')

## 8. Escuchar los segmentos

Cada card muestra: N°, timestamps, caracteres, el texto y el player de audio.

In [ ]:
from IPython.display import Audio, HTML, display
import soundfile as sf, base64, io

def _wav_to_b64(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

cards = []
for sub, path, err in results:
    text = sub.content.strip().replace('\n', ' ')
    start = str(sub.start).split('.')[0]
    end   = str(sub.end).split('.')[0]
    nchars = len(text)
    flag = '⚠️ CORTA' if nchars <= 30 else ''

    if err:
        card = f'''
        <div style="border:1px solid #c00;border-radius:6px;padding:10px;margin:6px 0;background:#fff0f0">
          <b>#{sub.index}</b> &nbsp; {start} → {end} &nbsp; ({nchars} chars) &nbsp; <span style="color:red">FALLÓ: {err}</span><br>
          <span style="color:#555">{text}</span>
        </div>'''
    else:
        b64 = _wav_to_b64(path)
        audio_data, sr = sf.read(path)
        dur = len(audio_data) / sr
        bg = '#fffbe6' if nchars <= 30 else '#f8f8f8'
        card = f'''
        <div style="border:1px solid #ccc;border-radius:6px;padding:10px;margin:6px 0;background:{bg}">
          <b>#{sub.index}</b> &nbsp; {start} → {end} &nbsp; ({nchars} chars, {dur:.2f}s TTS) &nbsp; <span style="color:#e60">{flag}</span><br>
          <span style="color:#333">{text}</span><br>
          <audio controls style="margin-top:6px;height:32px;width:100%">
            <source src="data:audio/wav;base64,{b64}" type="audio/wav">
          </audio>
        </div>'''
    cards.append(card)

html = f'<div style="font-family:sans-serif;font-size:13px">{chr(10).join(cards)}</div>'
display(HTML(html))

## 9. Re-generar un segmento puntual

Elegí el N° de segmento, ajustá la temperatura, y volvé a generarlo sin correr todo.

In [ ]:
import ipywidgets as W
from IPython.display import display, Audio, HTML
import soundfile as sf, base64

seg_n_w  = W.IntText(value=ACTIVE_SUBS[0].index if ACTIVE_SUBS else 1,
                     description='N° segmento:', style={'description_width':'initial'})
temp2_w  = W.FloatSlider(value=TEMPERATURE, min=0.1, max=1.0, step=0.05,
                         description='Temperatura:', style={'description_width':'initial'})
btn      = W.Button(description='Generar', button_style='primary')
out      = W.Output()

def _regen(_):
    out.clear_output()
    n = seg_n_w.value
    tmp = temp2_w.value
    sub = next((s for s in SUBS if s.index == n), None)
    if sub is None:
        with out: print(f'Segmento #{n} no encontrado en el SRT.')
        return
    text = sub.content.strip().replace('\n', ' ')
    out_path = OUT_DIR / f'seg_{n:04d}_t{int(tmp*100)}.wav'
    with out:
        print(f'Generando #{n} con temperatura {tmp}...')
        print(f'Texto: {text}')
    try:
        tts.tts_to_file(
            text=text,
            file_path=str(out_path),
            speaker_wav=VOICE_REF,
            language=LANG,
            split_sentences=False,
            temperature=tmp,
        )
        audio_data, sr = sf.read(out_path)
        dur = len(audio_data) / sr
        b64 = base64.b64encode(open(out_path,'rb').read()).decode()
        with out:
            print(f'Listo — {dur:.2f}s')
            display(HTML(f'<audio controls style="width:100%"><source src="data:audio/wav;base64,{b64}" type="audio/wav"></audio>'))
    except Exception as e:
        with out: print(f'ERROR: {e}')

btn.on_click(_regen)
display(seg_n_w, temp2_w, btn, out)

## 10. Descargar todos los WAV

In [ ]:
import shutil
from google.colab import files
zip_path = '/content/xtts_out.zip'
shutil.make_archive('/content/xtts_out', 'zip', OUT_DIR)
files.download(zip_path)

## Reset

Limpia los módulos cacheados y el repo clonado.

In [ ]:
import sys, shutil
for m in list(sys.modules):
    if m.startswith('batch_dub'):
        del sys.modules[m]
print('Módulos limpiados')
if os.path.exists('/content/Sta-RU'):
    shutil.rmtree('/content/Sta-RU')
    print('Repo eliminado')